# Last Letter Concatenation — Qwen2.5-7B 정밀 분석 (self-contained)

이 노트북은 **메인 실험 결과 없이도 단독으로 작동**합니다. vLLM 7B를 즉석 로드해 last_letter 300문제만 추론하고, 그 raw 응답을 메모리상 DataFrame으로 들고 있으면서 분석 결과를 결과창에 표/그래프로 띄웁니다.

### 소요 시간 (A100 40GB 기준)
- vLLM 설치: ~2분 (이미 설치돼있으면 skip)
- 7B 로드: ~3분
- 300문제 × (Standard + CoT) 추론: ~5~10분
- **총 약 10~15분**

### 분석 내용
1. Standard vs CoT 정확도 + 케이스 cross-tab
2. CoT만 정답인 케이스 raw 응답 비교 (CoT의 진짜 기여)
3. 오류 유형 자동 분류 (글자 수 부족 / 순서 / 추출 실패 등)
4. CoT reasoning chain 내부 step 정확도 (글자 추출 능력 vs 이어붙이기 능력 분리)
5. 입력 단어 특성과 정확도 상관


## 1. 환경 설정 (vLLM 설치 + V1 우회 패치)

vLLM 0.7+의 V1 엔진은 Jupyter `ipykernel` stdout과 호환성 문제(`io.UnsupportedOperation: fileno`)가 있으므로 V0 엔진 강제 + stdout monkey-patch 적용. **vLLM import 전에** 환경변수가 설정돼야 합니다.


In [ ]:
!pip install -q vllm 2>&1 | tail -3

In [ ]:
import os, sys, io

# (1) V1 엔진 비활성화 — vllm import 전 설정
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# (2) ipykernel OutStream의 fileno() 미지원 우회
class _RealFilenoStream(io.IOBase):
    def __init__(self, original_stream, real_fd):
        self._original = original_stream
        self._real_fd = real_fd
    def __getattr__(self, name): return getattr(self._original, name)
    def fileno(self): return self._real_fd
    def write(self, s): return self._original.write(s)
    def flush(self): return self._original.flush()
    def writable(self): return True
    def readable(self): return False
    def isatty(self):   return False

if not getattr(sys.stdout, "_real_fileno_patched", False):
    sys.stdout = _RealFilenoStream(sys.stdout, 1)
    sys.stdout._real_fileno_patched = True
if not getattr(sys.stderr, "_real_fileno_patched", False):
    sys.stderr = _RealFilenoStream(sys.stderr, 2)
    sys.stderr._real_fileno_patched = True

if "vllm" in sys.modules:
    print("⚠ vllm이 이미 import됨. V1 비활성화가 적용 안 됐을 수 있음 → 런타임 재시작 후 다시 실행하세요.")
else:
    print("✓ V1 비활성화 + stdout patch 적용")

## 2. Last Letter 데이터 300문제 생성 (in-memory)

논문 §3.3의 in-domain 설정(4단어) 그대로. 영어 이름 풀에서 4개 샘플 → 각 단어의 마지막 글자를 이어 붙임. seed=42로 재현 가능.


In [ ]:
import random, re, json
from dataclasses import dataclass
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 200)

SEED = 42
N_PROBLEMS = 300

_NAMES = [
    "James","Mary","John","Patricia","Robert","Jennifer","Michael","Linda",
    "William","Elizabeth","David","Barbara","Richard","Susan","Joseph","Jessica",
    "Thomas","Sarah","Charles","Karen","Christopher","Nancy","Daniel","Lisa",
    "Matthew","Margaret","Anthony","Betty","Mark","Sandra","Donald","Ashley",
    "Steven","Kimberly","Paul","Emily","Andrew","Donna","Joshua","Michelle",
    "Kenneth","Carol","Kevin","Amanda","Brian","Melissa","George","Deborah",
    "Edward","Stephanie","Ronald","Dorothy","Timothy","Rebecca","Jason","Sharon",
    "Jeffrey","Laura","Ryan","Cynthia","Jacob","Amy","Gary","Kathleen",
]

def generate_problems(n, seed):
    rng = random.Random(seed)
    out = []
    for i in range(n):
        names = rng.sample(_NAMES, 4)
        q = f'Take the last letters of the words in "{" ".join(names)}" and concatenate them.'
        ans = "".join(name[-1].lower() for name in names)
        out.append({"idx": i, "question": q, "gold": ans, "words": names})
    return out

problems = generate_problems(N_PROBLEMS, SEED)
print(f"{len(problems)}문제 생성. 첫 3개 sample:")
for p in problems[:3]:
    print(f"  Q: {p['question'][:80]}...  →  A: {p['gold']}")

## 3. 프롬프트 — 논문 Appendix G Table 21 (4-shot)

논문의 in-context exemplar를 그대로 옮김. CoT 조건은 reasoning chain 포함, Standard 조건은 최종 답만.


In [ ]:
_EXEMPLARS = [
    ('Take the last letters of the words in "Elon Musk" and concatenate them.',
     'The last letter of "Elon" is "n". The last letter of "Musk" is "k". '
     'Concatenating them is "nk". The answer is nk.',
     "nk"),
    ('Take the last letters of the words in "Larry Page" and concatenate them.',
     'The last letter of "Larry" is "y". The last letter of "Page" is "e". '
     'Concatenating them is "ye". The answer is ye.',
     "ye"),
    ('Take the last letters of the words in "Sergey Brin" and concatenate them.',
     'The last letter of "Sergey" is "y". The last letter of "Brin" is "n". '
     'Concatenating them is "yn". The answer is yn.',
     "yn"),
    ('Take the last letters of the words in "Bill Gates" and concatenate them.',
     'The last letter of "Bill" is "l". The last letter of "Gates" is "s". '
     'Concatenating them is "ls". The answer is ls.',
     "ls"),
]

def build_prompt(question: str, condition: str) -> str:
    parts = []
    for q, rat, fin in _EXEMPLARS:
        ans_text = rat if condition == "cot" else f"The answer is {fin}."
        parts.append(f"Q: {q}\nA: {ans_text}")
    parts.append(f"Q: {question}\nA:")
    return "\n\n".join(parts)

print("=== CoT 프롬프트 (마지막 exemplar만 표시) ===")
print(build_prompt(problems[0]["question"], "cot").split("\n\n")[-2])
print("\n=== Standard 프롬프트 (마지막 exemplar만 표시) ===")
print(build_prompt(problems[0]["question"], "standard").split("\n\n")[-2])

## 4. Qwen2.5-7B 로드 후 두 조건 추론

A100 40GB에서 fp16 적재 (~14GB). max_model_len=4096이면 8-shot 프롬프트 + 응답 충분.

⏱ 이 셀이 가장 오래 걸립니다 (로드 ~3분 + 추론 ~5-10분).


In [ ]:
from vllm import LLM, SamplingParams
import gc

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    trust_remote_code=True,
)
tok = llm.get_tokenizer()

def _chat(p):
    return tok.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False, add_generation_prompt=True,
    )

# Standard 추론 (greedy)
std_prompts = [_chat(build_prompt(p["question"], "standard")) for p in problems]
sp_greedy = SamplingParams(temperature=0.0, max_tokens=512)
std_outs = llm.generate(std_prompts, sp_greedy, use_tqdm=True)
std_raw = [o.outputs[0].text for o in std_outs]

# CoT 추론 (greedy)
cot_prompts = [_chat(build_prompt(p["question"], "cot")) for p in problems]
cot_outs = llm.generate(cot_prompts, sp_greedy, use_tqdm=True)
cot_raw = [o.outputs[0].text for o in cot_outs]

# 메모리 해제
del llm
gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except Exception:
    pass

print(f"\n추론 완료. std_raw {len(std_raw)}개, cot_raw {len(cot_raw)}개")

## 5. 답 추출 (regex) + 정답 여부 라벨링

논문 방식: `The answer is X` 패턴을 1순위, 실패 시 인용부호로 감싸인 알파벳 토큰을 fallback. `pred`와 `gold`를 비교해 `correct` 플래그.


In [ ]:
_ANSWER_IS_RE = re.compile(r"[Tt]he answer is\s*:?\s*(.+?)(?:[\.\n]|$)", re.DOTALL)

def extract_last_letter(text: str):
    m = _ANSWER_IS_RE.search(text)
    if m:
        tail = m.group(1).strip().strip('"').strip("'").strip(".")
        only_alpha = re.sub(r"[^a-zA-Z]", "", tail)
        if only_alpha:
            return only_alpha.lower(), "primary"
    quoted = re.findall(r'"([a-zA-Z]+)"', text)
    if quoted:
        return quoted[-1].lower(), "fallback_quoted"
    return "", "failed"

def normalize(s):
    return (s or "").strip().lower()

def build_df(problems_, raw_):
    rows = []
    for p, r in zip(problems_, raw_):
        val, method = extract_last_letter(r)
        rows.append({
            "idx": p["idx"],
            "question": p["question"],
            "gold": p["gold"],
            "words": p["words"],
            "pred": val,
            "method": method,
            "raw": r,
        })
    df = pd.DataFrame(rows)
    df["correct"] = df.apply(lambda r: normalize(r["pred"]) == normalize(r["gold"]), axis=1)
    return df

std = build_df(problems, std_raw)
cot = build_df(problems, cot_raw)

print(f"Standard: {std['correct'].sum():3d}/{len(std)} ({std['correct'].mean()*100:5.2f}%) 정답")
print(f"CoT     : {cot['correct'].sum():3d}/{len(cot)} ({cot['correct'].mean()*100:5.2f}%) 정답")
print(f"\n추출 실패율 - Standard: {(std['method']=='failed').mean()*100:.1f}%, "
      f"CoT: {(cot['method']=='failed').mean()*100:.1f}%")

## 6. 케이스 cross-tab

같은 문제에 대해 두 조건의 정답 여부를 교차표로:
- **양쪽 정답** / **양쪽 오답** / **Standard만** / **CoT만 정답** ⭐


In [ ]:
m = std.merge(cot, on="idx", suffixes=("_std", "_cot"))
m["gold"]     = m["gold_std"]
m["question"] = m["question_std"]

ct = pd.crosstab(m["correct_std"], m["correct_cot"],
                 rownames=["Standard 정답?"], colnames=["CoT 정답?"])
print("=== 케이스 분포 ===")
display(ct)

both_right = m[ m.correct_std &  m.correct_cot]
both_wrong = m[~m.correct_std & ~m.correct_cot]
std_only   = m[ m.correct_std & ~m.correct_cot]
cot_only   = m[~m.correct_std &  m.correct_cot]

print(f"\n양쪽 정답   : {len(both_right):3d}")
print(f"Standard만  : {len(std_only):3d}  (CoT가 망친 경우)")
print(f"CoT만 정답  : {len(cot_only):3d}  ⭐ CoT의 진짜 기여")
print(f"양쪽 오답   : {len(both_wrong):3d}  (7B로는 못 푸는 문제)")

## 7. ⭐ CoT만 정답인 케이스 — raw 응답 비교

CoT가 어떻게 답에 도달했는지 직접 봅니다.


In [ ]:
def _trim(s, n=300):
    s = (s or "").replace("\n", " ⏎ ").strip()
    return s if len(s) <= n else s[:n] + " ...[truncated]"

view = cot_only[["question", "gold", "pred_std", "pred_cot"]].copy()
view["raw_std_short"] = cot_only["raw_std"].apply(_trim)
view["raw_cot_short"] = cot_only["raw_cot"].apply(_trim)
display(view.head(10))

## 8. 양쪽 오답 케이스 (7B의 한계)

CoT로도 못 푼 문제는 무엇이 모델을 헷갈리게 했는지.


In [ ]:
view2 = both_wrong[["question", "gold", "pred_std", "pred_cot"]].copy()
view2["raw_cot_short"] = both_wrong["raw_cot"].apply(lambda r: _trim(r, 350))
display(view2.head(8))

## 9. 오류 유형 자동 분류

- `correct`: 정답
- `extraction_failed`: 빈 답 (추출 실패)
- `length_short` / `length_long`: 글자 수 부족/초과
- `anagram`: 글자 구성은 동일하나 순서가 다름 (글자는 뽑았으나 concat 순서 실수)
- `partial_match`: 1글자 빼고 다 맞음
- `wrong_letters`: 글자 자체가 다름


In [ ]:
def classify_error(pred, gold):
    p, g = normalize(pred), normalize(gold)
    if p == g: return "correct"
    if not p: return "extraction_failed"
    if len(p) != len(g):
        return "length_short" if len(p) < len(g) else "length_long"
    if sorted(p) == sorted(g): return "anagram"
    overlap = sum(1 for c in p if c in g)
    if overlap >= len(g) - 1: return "partial_match"
    return "wrong_letters"

std["err"] = std.apply(lambda r: classify_error(r["pred"], r["gold"]), axis=1)
cot["err"] = cot.apply(lambda r: classify_error(r["pred"], r["gold"]), axis=1)

err_order = ["correct", "anagram", "partial_match", "length_short",
             "length_long", "wrong_letters", "extraction_failed"]
err_dist = pd.DataFrame({
    "Standard": std["err"].value_counts(),
    "CoT":      cot["err"].value_counts(),
}).reindex(err_order).fillna(0).astype(int)
err_dist["Std %"] = (err_dist["Standard"] / len(std) * 100).round(1)
err_dist["CoT %"] = (err_dist["CoT"]      / len(cot) * 100).round(1)
print("=== 오류 유형 분포 ===")
display(err_dist)

In [ ]:
# 막대그래프
plot_df = err_dist[["Standard", "CoT"]].drop("correct", errors="ignore").reset_index()
plot_df = plot_df.rename(columns={"index": "err"})
melted = plot_df.melt(id_vars="err", var_name="condition", value_name="count")

plt.figure(figsize=(10, 4))
sns.barplot(data=melted, x="err", y="count", hue="condition")
plt.title("Last Letter (Qwen2.5-7B) — 오류 유형 분포 (correct 제외)")
plt.ylabel("문항 수"); plt.xlabel("오류 유형")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 10. ⭐ CoT reasoning chain 내부 step 분석

CoT 응답은 보통 단어별로 last letter를 명시:
```
The last letter of "James" is "s".
The last letter of "Robert" is "t".
Concatenating them is "st...". The answer is st....
```

여기서 두 가지를 분리:
- **Step 정확도**: 각 단어의 마지막 글자를 모델이 올바르게 추출하는가?
- **Concat 정확도**: step이 다 맞아도 마지막 연결에서 틀리는가? (working memory 한계 시그널)


In [ ]:
STEP_RE = re.compile(
    r'''last\s+letter\s+of\s+["']?([A-Za-z]+)["']?\s+is\s+["']?([A-Za-z])["']?''',
    re.IGNORECASE,
)

def extract_steps(raw):
    return [(w, l) for w, l in STEP_RE.findall(raw or "")]

cot["steps"]        = cot["raw"].apply(extract_steps)
cot["n_steps_out"]  = cot["steps"].apply(len)

print("응답 step 추출 수 분포:", Counter(cot["n_steps_out"]))

In [ ]:
def step_accuracy(row):
    correct = sum(1 for w, l in row["steps"] if w[-1].lower() == l.lower())
    return correct, len(row["steps"])

cot[["step_correct", "step_total"]] = cot.apply(
    lambda r: pd.Series(step_accuracy(r)), axis=1
)
cot["all_steps_correct"] = (cot["step_correct"] == cot["step_total"]) & (cot["step_total"] > 0)

ts, cs = int(cot["step_total"].sum()), int(cot["step_correct"].sum())
print(f"전체 step accuracy: {cs}/{ts} = {cs/max(ts,1)*100:.1f}%")

m_all = cot[cot["all_steps_correct"]]
breakdown = pd.Series({
    "step 다 맞음 & 최종 정답"    : int((m_all["correct"]).sum()),
    "step 다 맞음 & 최종 오답"    : int((~m_all["correct"]).sum()),
    "step 일부 틀림 & 최종 정답"  : int((cot[~cot["all_steps_correct"] &  cot["correct"]]).shape[0]),
    "step 일부 틀림 & 최종 오답"  : int((cot[~cot["all_steps_correct"] & ~cot["correct"]]).shape[0]),
}, name="문항 수")
print("\n=== CoT 내부 step ↔ 최종 답 일치 분해 ===")
display(breakdown.to_frame())
print(
    "\n해석:\n"
    "  - 'step 다 맞음 & 최종 오답'이 크면 → 글자는 뽑지만 concat에서 실수 (working memory)\n"
    "  - 'step 일부 틀림 & 최종 정답'은 보통 매우 적음"
)

In [ ]:
# 단어 position별 step 정확도
pos = []
for _, row in cot.iterrows():
    for i, (w, l) in enumerate(row["steps"][:4]):
        pos.append({"position": i, "correct": int(w[-1].lower() == l.lower())})
pos_df = pd.DataFrame(pos)
if len(pos_df):
    by_pos = pos_df.groupby("position")["correct"].agg(["mean", "count"])
    by_pos["accuracy %"] = (by_pos["mean"] * 100).round(1)
    by_pos = by_pos[["accuracy %", "count"]]
    print("=== 단어 position별 step 정확도 ===")
    display(by_pos)
    print("(position 0 = 첫 단어, position 3 = 마지막 단어)")

## 11. ⭐ "step 다 맞음 & 최종 오답" 케이스 직접 보기

이 카테고리가 클수록 7B의 한계는 **"글자는 안다 / 이어붙이기를 못 한다"** — working memory 부족.


In [ ]:
tricky = cot[cot["all_steps_correct"] & ~cot["correct"]].copy()
print(f"카테고리 문항 수: {len(tricky)}")

if len(tricky):
    tv = tricky[["question", "gold", "pred", "steps"]].copy()
    tv["model_letters"] = tricky["steps"].apply(lambda s: "".join(l.lower() for _, l in s))
    tv["raw_short"] = tricky["raw"].apply(lambda r: _trim(r, 350))
    display(tv.head(10))
else:
    print("비어있음 → 7B에서 글자 추출과 concat 능력이 거의 동기화됨.")

## 12. 입력 변수 vs 정확도

단어 길이, 글자 등이 정확도에 영향을 주는지.


In [ ]:
cot["avg_word_len"] = cot["words"].apply(lambda ws: float(np.mean([len(w) for w in ws])))
std["avg_word_len"] = std["words"].apply(lambda ws: float(np.mean([len(w) for w in ws])))

def acc_by_bin(df, col, bins):
    df = df.copy()
    df["bin"] = pd.cut(df[col], bins=bins)
    g = df.groupby("bin", observed=True)["correct"].agg(["mean", "count"])
    g["accuracy %"] = (g["mean"]*100).round(1)
    return g[["accuracy %", "count"]]

print("=== 평균 단어 길이별 CoT 정확도 ===")
display(acc_by_bin(cot, "avg_word_len", bins=[0,5,6,7,8,20]))
print("\n=== 평균 단어 길이별 Standard 정확도 ===")
display(acc_by_bin(std, "avg_word_len", bins=[0,5,6,7,8,20]))

In [ ]:
# '진짜 마지막 글자'별 step 정확도 — tokenizer 영향 의심 지점 찾기
letter_stat = {}
for _, row in cot.iterrows():
    for w, l in row["steps"]:
        tl = w[-1].lower()
        letter_stat.setdefault(tl, [0, 0])
        letter_stat[tl][1] += 1
        if l.lower() == tl:
            letter_stat[tl][0] += 1

ldf = pd.DataFrame(
    [(k, v[0], v[1], v[0]/v[1]*100 if v[1] else 0) for k, v in letter_stat.items()],
    columns=["true_last_letter", "correct", "total", "step_acc %"],
).sort_values("total", ascending=False)
ldf["step_acc %"] = ldf["step_acc %"].round(1)
print("=== '진짜 마지막 글자'별 step 정확도 (상위 빈도순) ===")
display(ldf.head(15))

## 13. 7B 분석 정리 (체크리스트)

위 셀 출력을 보고 자신에게 답해 보세요:

1. **§6 cross-tab**: CoT만 정답인 문항이 몇 개? 이게 곧 "CoT가 만든 7B의 reasoning capacity".
2. **§9 오류 분포**: Standard의 오류가 `length_short`/`length_long`에 몰리면 → 한 번에 풀려다가 글자 수 자체가 흐트러진다.
3. **§10 전체 step accuracy**: 90%+ 라면 7B는 "글자 뽑는 능력"은 충분. 그럼에도 최종 24%인 이유는?
4. **§10~11 'step 다 맞음 & 최종 오답' 비중**: 크면 → 글자는 알지만 4개 순서대로 들고 있을 working memory 부족. 32B의 67% 점프 정체가 이것일 가능성.
5. **§12 입력 특성**: 단어 길이별 정확도 차이가 크면 long-tail token 영향. 글자별 step acc가 비대칭이면 tokenizer 영향(예: word-final 'y') 의심.
